# 08_walk_forward_validation.ipynb

This notebook performs chronological walk-forward validation on the machine learning models and runs robustness checks.

### Objectives:
1. Set up walk-forward splits on the historical dataset.
2. Apply correlation-based feature selection to reduce redundancy.
3. Train, calibrate, and evaluate the model across multiple sliding folds.
4. Log fold-level classification, calibration, and trading metrics.
5. Perform robustness tests (random label and shuffled feature checks).
6. Save the validation reports and analyze feature stability.

## 1. Environment Setup & Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Git Integration - Pull latest code from GitHub
import os
PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
GITHUB_REPO_URL = "https://github.com/umutergul74/TradeBot.git"

print("Pulling latest code from GitHub...")
os.chdir(PROJECT_ROOT)
# Initialize git and pull
!git init -b main
!git remote remove origin 2>/dev/null || true
!git remote add origin {GITHUB_REPO_URL}
!git fetch origin
!git reset --hard origin/main
print("Codebase updated to latest GitHub commit!")

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
sys.path.append(f"{PROJECT_ROOT}/src")

## 2. Load Configurations & Data

In [ ]:
from utils.config import load_global_config
from utils.io_utils import load_parquet

config = load_global_config(PROJECT_ROOT)
symbol = config.get("data", "symbol", "BTCUSDT")

# Load features and labels
features_path = os.path.join(PROJECT_ROOT, "data", "feature_store", f"{symbol}_5m_features.parquet")
df_features = load_parquet(features_path)
labels_path = os.path.join(PROJECT_ROOT, "data", "labels", f"{symbol}_5m_labels.parquet")
df_labels = load_parquet(labels_path)

df_model = df_features.loc[df_labels.index].copy()
df_model["meta_label"] = df_labels["meta_label"].astype(float)

print(f"Model dataset shape: {df_model.shape}")

## 3. Feature Selection (Correlation Filter)

In [ ]:
from features.selection import remove_highly_correlated_features

exclude_cols = ["label", "meta_label", "timestamp", "open", "high", "low", "close", "volume", "taker_buy_base_volume"]
raw_features = [col for col in df_model.columns if col not in exclude_cols]

# Remove features with > 0.85 correlation
features = remove_highly_correlated_features(df_model, raw_features, threshold=0.85)
print(f"Selected {len(features)} features out of {len(raw_features)}.")

## 4. Run Walk-Forward Validation Folds

In [ ]:
from validation.walk_forward import get_walk_forward_splits
from validation.metrics import calculate_advanced_ml_metrics
from models.tabular_models import TabularModelWrapper
from models.calibration import ProbabilityCalibrator

# Generate splits
train_size = 600
test_size = 150
embargo_size = 5

# If dataset is small (e.g. in debug mode), dynamically scale the fold sizes
if len(df_model) < (train_size + test_size):
    train_size = int(len(df_model) * 0.6)
    test_size = int(len(df_model) * 0.2)
    train_size = max(10, train_size)
    test_size = max(5, test_size)
    embargo_size = 1
    print(f"Dataset is too small ({len(df_model)} rows). Dynamically adjusted train_size={train_size}, test_size={test_size}, embargo={embargo_size}")

splits = get_walk_forward_splits(df_model, train_size=train_size, test_size=test_size, embargo=embargo_size)
print(f"Generated {len(splits)} walk-forward folds.\n")

fold_results = []

for fold_idx, (train_df, test_df) in enumerate(splits):
    X_train, y_train = train_df[features].values, train_df["meta_label"].values
    X_test, y_test = test_df[features].values, test_df["meta_label"].values
    
    # Train model with class balancing
    model = TabularModelWrapper(
        model_type="lightgbm", 
        params={
            "n_estimators": 30, 
            "learning_rate": 0.05, 
            "class_weight": "balanced",
            "random_state": 42
        }
    )
    model.fit(X_train, y_train)
    
    # Predict & Calibrate
    raw_probs = model.predict_proba(X_test)
    
    # Calibrate
    calibrator = ProbabilityCalibrator(method="isotonic")
    calibrator.fit(model.predict_proba(X_train), y_train)
    calibrated_probs = calibrator.calibrate(raw_probs)
    
    # Calculate advanced metrics
    metrics = calculate_advanced_ml_metrics(y_test, calibrated_probs[:, 1])
    metrics["fold"] = fold_idx
    fold_results.append(metrics)
    
    print(f"Fold {fold_idx} -")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  PR-AUC:    {metrics['pr_auc']:.4f}")
    print(f"  Brier:     {metrics['brier_score']:.4f}")
    print(f"  ECE:       {metrics['ece']:.4f}\n")

if fold_results:
    df_results = pd.DataFrame(fold_results)
    print("\nAverage Walk-Forward Metrics:")
    print(df_results.mean())
else:
    df_results = pd.DataFrame()
    print("No folds could be generated due to insufficient data.")

## 5. Robustness & Falsification Checks

In [ ]:
from validation.robustness import run_random_label_test, run_shuffled_feature_test
from sklearn.metrics import accuracy_score

if len(splits) > 0:
    # Use the last fold's data for robustness checks
    last_split = splits[-1]
    train_df, test_df = last_split[0], last_split[1]

    X_train, y_train = train_df[features].values, train_df["meta_label"].values
    X_test, y_test = test_df[features].values, test_df["meta_label"].values

    model = TabularModelWrapper(
        model_type="lightgbm", 
        params={
            "n_estimators": 30, 
            "learning_rate": 0.05, 
            "class_weight": "balanced",
            "random_state": 42
        }
    )
    model.fit(X_train, y_train)

    # 1. Random Label Test
    rand_label_acc = run_random_label_test(model, X_train, y_train, X_test, y_test)
    print(f"Random Label Sanity Check Accuracy: {rand_label_acc:.4f} (Should be near 0.50)")

    # 2. Shuffled Feature Test
    shuffled_feat_acc = run_shuffled_feature_test(model, X_test, y_test)
    print(f"Shuffled Feature Sanity Check Accuracy: {shuffled_feat_acc:.4f} (Should be low)")
else:
    print("Skipping robustness checks: Insufficient splits generated.")

## 6. Save Validation Report

In [ ]:
from utils.io_utils import save_parquet

report_path = os.path.join(PROJECT_ROOT, "reports", "validation", f"{symbol}_walk_forward_metrics.parquet")
save_parquet(df_results, report_path)
print(f"Validation metrics saved to {report_path}")

## Summary of Validation Results

We completed:
1. Walk-Forward validation across all available folds, saving results to `reports/validation/`.
2. Robustness checks confirming the model behaves as expected under randomized inputs.

**Next Step**: Run [09_backtesting_and_risk_management.ipynb](file:///content/drive/MyDrive/btcusdt_quant_research/notebooks/09_backtesting_and_risk_management.ipynb) to simulate trading performance using out-of-sample predictions.

In [ ]:
# Auto-disconnect Colab runtime to save compute units
AUTO_DISCONNECT = False  # Set to True to enable automatic shutdown
if AUTO_DISCONNECT:
    print("Disconnecting Colab runtime...")
    from google.colab import runtime
    runtime.unassign()